# Hierarchical System Orchestration

This notebook runs through the entire hierarchical system:
1. **System 1**: Property Extraction (runs once)
2. **System 2**: Material Discovery → **System 3**: Manufacturing Assessment
3. If System 3 rejects materials (status="blocked"), loops back to System 2 (up to 3 iterations)
4. Stops when System 3 returns manufacturable candidate or max iterations reached

**Workflow**: 
```
System 1 → System 2 → System 3
              ↓         ↓
           (if blocked) ↓
              ←─────────┘
           (loop up to 3 times)
```

## Setup: Import modules and load configuration

In [1]:
import sys
import os
import json
import glob
import uuid
import time
from pathlib import Path
from datetime import datetime, timedelta

# Set project_root to workspace root
current_dir = Path().resolve()
if (current_dir / "src").exists() and (current_dir / "config").exists():
    project_root = current_dir
elif (current_dir.parent / "src").exists() and (current_dir.parent / "config").exists():
    project_root = current_dir.parent
else:
    # Fallback: check SYS3DEV_ROOT env var, otherwise assume parent of notebooks/
    import os
    project_root = Path(os.environ.get("SYS3DEV_ROOT", current_dir.parent))
sys.path.insert(0, str(project_root))
print(f"project_root set to: {project_root}")

# Import modules
from src.config import load_config
from src.utils import (
    llm, TransformerEmbeddingFunction, ChatLogger, MaterialDatabase, 
    PropertyMapper, SubgraphProcessor, MaterialGrounding, Step1Cache,
    save_evaluation_export
)
from src.agents import (
    ResearchAnalyst, ResearchManager, ResearchAssistant, ResearchScientist,
    RejectedCandidateTracker, MaterialScientist, MultiAnalyst
)
from src.pipelines import (
    run_fixed_pipeline,
    run_material_discovery_pipeline,
    run_manufacturability_assessment_pipeline
)

# ChromaDB and Graph imports
from chromadb import PersistentClient
from chromadb.config import Settings
from autogen.agentchat.contrib.vectordb.chromadb import ChromaVectorDB
import networkx as nx
from sentence_transformers import SentenceTransformer
from GraphReasoning import load_embeddings

# Disable tqdm progress bars globally to prevent clutter in notebook output
os.environ['TQDM_DISABLE'] = '1'
# Also disable tqdm by monkey-patching the tqdm class
import tqdm
original_tqdm_init = tqdm.tqdm.__init__
def disabled_tqdm_init(self, *args, **kwargs):
    kwargs['disable'] = True
    return original_tqdm_init(self, *args, **kwargs)
tqdm.tqdm.__init__ = disabled_tqdm_init

# Load configuration
config = load_config()
print("✓ Configuration loaded")

project_root set to: /orcd/pool/004/tphage/SG_march/MARS
✓ Configuration loaded


## Initialize LLM and Embeddings

In [2]:
# Initialize LLM wrapper
llm_config = {
    "api_key": config["llm"]["api_key"],
    "base_url": config["llm"]["base_url"],
    "model": config["llm"]["model_name"],
    "max_tokens": config["llm"]["max_tokens"]
}
llm_instance = llm(llm_config)
generate = llm_instance.generate_cli
print("✓ LLM wrapper initialized")

# Initialize embedding model
embedding_tokenizer = ''
embedding_model = SentenceTransformer(config["embeddings"]["model_name"], trust_remote_code=True)
embedding_function = TransformerEmbeddingFunction(
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model
)
print("✓ Embedding model initialized")

✓ LLM wrapper initialized


<All keys matched successfully>


✓ Embedding model initialized


## Load Knowledge Graphs and Embeddings

In [3]:
# Load Knowledge Graphs
graphs_cfg = config["data"]["graphs"]
kg_dir = graphs_cfg["kg_dir"]

# Material Properties Knowledge Graph (used in System 1 Step 7 and System 2)
mp_cfg = graphs_cfg["material_properties"]
mp_graph_path = os.path.join(kg_dir, mp_cfg["graph_file"])
G_materialproperties = nx.read_graphml(mp_graph_path)
relation = nx.get_edge_attributes(G_materialproperties, "title")
nx.set_edge_attributes(G_materialproperties, relation, "relation")
nx.set_node_attributes(G_materialproperties, nx.pagerank(G_materialproperties), "pr")
print(f"Material Properties KG loaded: {G_materialproperties}")

# Load node embeddings for Material Properties KG
mp_embeddings_path = os.path.join(kg_dir, mp_cfg["embedding_file"])
node_embeddings_materialproperties = load_embeddings(mp_embeddings_path)
print("✓ Material Properties knowledge graph and embeddings loaded")

# PFAS Knowledge Graph (used in System 1 Step 4)
pfas_cfg = graphs_cfg["pfas"]
pfas_graph_path = os.path.join(kg_dir, pfas_cfg["graph_file"])
G_pfas = nx.read_graphml(pfas_graph_path)
relation_pfas = nx.get_edge_attributes(G_pfas, "title")
nx.set_edge_attributes(G_pfas, relation_pfas, "relation")
nx.set_node_attributes(G_pfas, nx.pagerank(G_pfas), "pr")
print(f"PFAS KG loaded: {G_pfas}")

# Load node embeddings for PFAS KG
pfas_embeddings_path = os.path.join(kg_dir, pfas_cfg["embedding_file"])
node_embeddings_pfas = load_embeddings(pfas_embeddings_path)
print("✓ PFAS knowledge graph and embeddings loaded")

# Patent Knowledge Graph (used in System 2)
patent_cfg = graphs_cfg["patents"]
patent_kg_path = os.path.join(kg_dir, patent_cfg["graph_file"])
G_patents = nx.read_graphml(patent_kg_path)
relation_patents = nx.get_edge_attributes(G_patents, "title")
nx.set_edge_attributes(G_patents, relation_patents, "relation")
nx.set_node_attributes(G_patents, nx.pagerank(G_patents), "pr")
print(f"Patent KG loaded: {G_patents}")

# Load Patent KG node embeddings
patent_embeddings_path = os.path.join(kg_dir, patent_cfg["embedding_file"])
node_embeddings_patents = load_embeddings(patent_embeddings_path)
print(f"✓ Loaded Patent KG embeddings: {len(node_embeddings_patents)} nodes")
print("✓ All knowledge graphs and embeddings loaded")

Material Properties KG loaded: DiGraph with 317800 nodes and 774009 edges
✓ Material Properties knowledge graph and embeddings loaded
PFAS KG loaded: DiGraph with 144335 nodes and 459248 edges
✓ PFAS knowledge graph and embeddings loaded
Patent KG loaded: DiGraph with 199008 nodes and 578361 edges
✓ Loaded Patent KG embeddings: 199008 nodes
✓ All knowledge graphs and embeddings loaded


In [4]:
# Initialize Step 1 cache to avoid re-running expensive operations in subsequent iterations
# Cache will store material grounding, KG extraction, and subgraph processing results
step1_cache = Step1Cache(enable_persistence=False)  # Set to True to persist across sessions
print("✓ Step 1 cache initialized")
cache_stats = step1_cache.get_stats()
print(f"  Cache entries: {cache_stats['memory_entries']}")

✓ Step 1 cache initialized
  Cache entries: 0


## Load ChromaDB Corpora

In [5]:
# Connect to ChromaDB databases
chroma_cfg = config["data"]["chromadb"]
base_path = chroma_cfg.get("base_path", "")

# PFAS ChromaDB (for System 1)
pfas_cfg = chroma_cfg["pfas"]
if base_path:
    pfas_db_path = os.path.join(base_path, pfas_cfg["database_path"])
else:
    pfas_db_path = pfas_cfg["database_path"]
client_pfas = PersistentClient(path=pfas_db_path)
pfas_collection = client_pfas.get_collection(pfas_cfg["collection_name"], embedding_function=embedding_function)
ChromaDB_PFAS = ChromaVectorDB(client=client_pfas, embedding_function=embedding_function)
ChromaDB_PFAS.active_collection = pfas_collection
print("✓ PFAS ChromaDB loaded")

# Patents ChromaDB (for System 2)
patents_cfg = chroma_cfg["patents"]
if base_path:
    patents_db_path = os.path.join(base_path, patents_cfg["database_path"])
else:
    patents_db_path = patents_cfg["database_path"]
client_patents = PersistentClient(path=patents_db_path)
patents_collections = client_patents.list_collections()
patents_collection_name = patents_cfg["collection_name"] or patents_collections[0].name
patents_collection = client_patents.get_collection(patents_collection_name, embedding_function=embedding_function)
print("✓ Patents ChromaDB loaded")

# MaterialDB ChromaDB (for System 2)
materialdb_cfg = chroma_cfg["materialdb"]
if base_path:
    materialdb_db_path = os.path.join(base_path, materialdb_cfg["database_path"])
else:
    materialdb_db_path = materialdb_cfg["database_path"]
client_materialdb = PersistentClient(path=materialdb_db_path)
materialdb_collections = client_materialdb.list_collections()
materialdb_collection_name = materialdb_cfg["collection_name"] or materialdb_collections[0].name
materialdb_collection = client_materialdb.get_collection(materialdb_collection_name, embedding_function=embedding_function)
print("✓ MaterialDB ChromaDB loaded")

# Manufacturing Textbooks ChromaDB (for System 3)
manufacturing_cfg = chroma_cfg.get("manufacturing_textbooks", {})
process_analysts = {}
if manufacturing_cfg:
    try:
        mfg_db_path = os.path.join(base_path, manufacturing_cfg["database_path"]) if base_path else manufacturing_cfg["database_path"]
        if os.path.exists(mfg_db_path):
            client_mfg = PersistentClient(path=mfg_db_path)
            mfg_collections = client_mfg.list_collections()
            mfg_collection_name = manufacturing_cfg.get("collection_name") or (mfg_collections[0].name if mfg_collections else None)
            if mfg_collection_name:
                mfg_collection = client_mfg.get_collection(mfg_collection_name, embedding_function=embedding_function)
                n_results_per_source = config.get("pipelines", {}).get("manufacturability_assessment", {}).get("n_results_per_source", 5)
                process_analysts["manufacturing_textbooks"] = ResearchAnalyst(
                    collection=mfg_collection,
                    embedding_function=embedding_function,
                    n_results=n_results_per_source,
                )
                print("✓ Manufacturing Textbooks ChromaDB loaded")
    except Exception as e:
        print(f"⚠ Skipping manufacturing_textbooks: {e}")

# Spec Sheets ChromaDB (for System 3, optional)
spec_sheets_cfg = chroma_cfg.get("spec_sheets", {})
if spec_sheets_cfg:
    try:
        spec_db_path = os.path.join(base_path, spec_sheets_cfg["database_path"]) if base_path else spec_sheets_cfg["database_path"]
        if os.path.exists(spec_db_path):
            client_spec = PersistentClient(path=spec_db_path)
            spec_collections = client_spec.list_collections()
            spec_collection_name = spec_sheets_cfg.get("collection_name") or (spec_collections[0].name if spec_collections else None)
            if spec_collection_name:
                spec_collection = client_spec.get_collection(spec_collection_name, embedding_function=embedding_function)
                n_results_per_source = config.get("pipelines", {}).get("manufacturability_assessment", {}).get("n_results_per_source", 5)
                process_analysts["spec_sheets"] = ResearchAnalyst(
                    collection=spec_collection,
                    embedding_function=embedding_function,
                    n_results=n_results_per_source,
                )
                print("✓ Spec Sheets ChromaDB loaded")
    except Exception as e:
        print(f"⚠ Skipping spec_sheets: {e}")

# Add Patents and MaterialDB to process analysts for System 3
n_results_per_source = config.get("pipelines", {}).get("manufacturability_assessment", {}).get("n_results_per_source", 5)
process_analysts["patents"] = ResearchAnalyst(
    collection=patents_collection,
    embedding_function=embedding_function,
    n_results=n_results_per_source,
)
process_analysts["materialdb"] = ResearchAnalyst(
    collection=materialdb_collection,
    embedding_function=embedding_function,
    n_results=n_results_per_source,
)

# Create MultiAnalyst for System 3 process retrieval
process_analyst = MultiAnalyst(process_analysts) if process_analysts else None
print(f"✓ Process analyst initialized with {len(process_analysts)} sources: {list(process_analysts.keys())}")

✓ PFAS ChromaDB loaded
✓ Patents ChromaDB loaded
✓ MaterialDB ChromaDB loaded
✓ Manufacturing Textbooks ChromaDB loaded
✓ Process analyst initialized with 3 sources: ['manufacturing_textbooks', 'patents', 'materialdb']


In [6]:
# Helper function to generate human-readable run_id
def generate_run_id(output_dir, prefix="system1"):
    """Generate human-readable run_id: YYYYMMDDHH_N format"""
    now = datetime.now()
    base_id = now.strftime("%Y%m%d%H")  # YYYYMMDDHH
    
    # Find existing files with same base
    pattern = os.path.join(output_dir, f"{prefix}_{base_id}_*.json")
    existing_files = glob.glob(pattern)
    
    # Extract counters and find max
    max_counter = -1
    for file in existing_files:
        filename = os.path.basename(file)
        # Extract counter from filename: prefix_YYYYMMDDHH_N.json
        parts = filename.replace(".json", "").split("_")
        if len(parts) >= 3:
            try:
                counter = int(parts[-1])
                max_counter = max(max_counter, counter)
            except ValueError:
                pass
    
    # Increment counter
    counter = max_counter + 1
    return f"{base_id}_{counter}"

# Generate base run_id for this session
output_dir = os.path.join(project_root, "pipeline_logs")
os.makedirs(output_dir, exist_ok=True)
base_run_id = generate_run_id(output_dir, prefix="system1").split("_")[0]  # Just the timestamp part
print(f"✓ Base run_id for this session: {base_run_id}")

# Initialize System 1 components
pipeline_cfg_s1 = config["pipelines"]["material_requirements"]
chat_logger_s1 = ChatLogger(
    run_id=generate_run_id(output_dir, prefix="system1"),
    pipeline="material_requirements"
)

analyst_s1 = ResearchAnalyst(
    collection=pfas_collection,
    embedding_function=embedding_function,
    n_results=pipeline_cfg_s1["n_results"],
    chat_logger=chat_logger_s1
)

manager_s1 = ResearchManager(
    name="research_manager",
    system_message=None,
    generate_fn=generate,
    chat_logger=chat_logger_s1
)

research_assistant_s1 = ResearchAssistant(
    name="research_assistant",
    system_message=None,
    generate_fn=generate,
    chat_logger=chat_logger_s1
)

# Initialize ResearchScientist with Material Properties knowledge graph only (for System 1 Step 7)
scientist_s1 = ResearchScientist(
    knowledge_graph=G_materialproperties,
    node_embeddings=node_embeddings_materialproperties,
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
    algorithm="shortest",
    generate_fn=generate,   # needed for LLM-based node classification in Phase 1.6
    chat_logger=chat_logger_s1
)

# Initialize PFAS ResearchScientist for querying PFAS KG during question answering (System 1 Step 4)
pfas_scientist_s1 = ResearchScientist(
    knowledge_graph=G_pfas,
    node_embeddings=node_embeddings_pfas,
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
    algorithm="shortest",
    chat_logger=chat_logger_s1
)

print("✓ System 1 agents initialized")

# Initialize System 2 components
pipeline_cfg_s2 = config["pipelines"]["material_discovery"]

# Create two ResearchAnalyst instances for System 2 - one for each ChromaDB
analyst_patents_s2 = ResearchAnalyst(
    collection=patents_collection,
    embedding_function=embedding_function,
    n_results=5,
    chat_logger=None  # Will be set per iteration
)

analyst_materialdb_s2 = ResearchAnalyst(
    collection=materialdb_collection,
    embedding_function=embedding_function,
    n_results=5,
    chat_logger=None  # Will be set per iteration
)

# Initialize ResearchScientist with BOTH knowledge graphs using "separate" strategy (for System 2)
scientist_s2 = ResearchScientist(
    knowledge_graph=G_materialproperties,
    node_embeddings=node_embeddings_materialproperties,
    embedding_tokenizer=embedding_tokenizer,
    embedding_model=embedding_model,
    algorithm="shortest",
    chat_logger=None,  # Will be set per iteration
    generate_fn=generate,
    # Second knowledge graph parameters
    knowledge_graph_2=G_patents,
    node_embeddings_2=node_embeddings_patents,
    kg_names=["material_properties", "patents"],
    kg_descriptions=[
        "Material properties and characteristics knowledge graph containing relationships between materials, properties, and applications",
        "Patent knowledge graph containing information about patents, materials, and related technologies"
    ],
    multi_kg_strategy="separate"
)

# Initialize new components for System 2 Step 1
property_mapper = PropertyMapper(
    embedding_model=embedding_model,
    embedding_tokenizer=embedding_tokenizer
)

# Load material database
material_db_cfg = config.get("data", {}).get("material_database", {})
material_db_path = material_db_cfg.get("path", "./data/internal_material_database.json")
material_db = MaterialDatabase.load_from_json(material_db_path, property_mapper=property_mapper)
print(f"✓ Material database loaded: {len(material_db)} materials")

# Initialize subgraph processor (uses config utils.subgraph_processor)
subgraph_processor = SubgraphProcessor(embedding_model=embedding_model)

# Initialize material grounding
material_grounding = MaterialGrounding(
    knowledge_graph=G_materialproperties,
    node_embeddings=node_embeddings_materialproperties,
    embedding_model=embedding_model,
    embedding_tokenizer=embedding_tokenizer
)

print("✓ System 2 components initialized")

# Initialize System 3 components (manager will be created per iteration with chat_logger)
print("✓ System 3 components initialized")
print("✓ All agents and components initialized")

✓ Base run_id for this session: 2026031908
✓ System 1 agents initialized
✓ ResearchScientist initialized with 2 knowledge graphs: ['material_properties', 'patents']
  Strategy: separate
  KG Descriptions: ['Material properties and characteristics knowledge graph containing relationships between materials, properties, and applications', 'Patent knowledge graph containing information about patents, materials, and related technologies']
✓ Material database loaded: 20 materials
✓ System 2 components initialized
✓ System 3 components initialized
✓ All agents and components initialized


In [7]:
# Initialize shared tracker that will be used across all System 2 → System 3 iterations
# This ensures rejected candidates from System 3 are automatically avoided in subsequent System 2 runs
tracker = RejectedCandidateTracker()
print("✓ Shared RejectedCandidateTracker initialized")
print(f"  Current rejected candidates: {len(tracker.get_all_rejected())}")

✓ Shared RejectedCandidateTracker initialized
  Current rejected candidates: 0


In [8]:
# Initialize PipelineRun tracking object
pipeline_start_time = datetime.utcnow()
pipeline_run_id = base_run_id  # Use the base_run_id from initialization

pipeline_run = {
    "pipeline_run_id": pipeline_run_id,
    "start_time": pipeline_start_time.isoformat() + "Z",
    "end_time": None,
    "total_duration_seconds": None,
    "system1": {
        "run_id": None,
        "start_time": None,
        "end_time": None,
        "duration_seconds": None,
        "result_path": None,
        "chat_log_path": None,
        "properties_extracted": None
    },
    "system2_system3_loop": {
        "max_iterations": None,
        "total_iterations": 0,
        "iterations": []
    },
    "final_outcome": {
        "status": None,
        "final_candidate": None,
        "total_rejected_candidates": None
    }
}

print("✓ PipelineRun tracking object initialized")
print(f"  Pipeline Run ID: {pipeline_run_id}")
print(f"  Start Time: {pipeline_start_time.isoformat()}Z")

✓ PipelineRun tracking object initialized
  Pipeline Run ID: 2026031908
  Start Time: 2026-03-19T12:59:36.535145Z


## Run System 1: Property Extraction

In [9]:
# Define your research query
sentence = "Find a substitute material for PFAS material THV (elastomeric terpolymer of tetrafluoroethylene, hexafluoropropylene, and vinylidene fluoride) that has broad chemical resistance, including acids, bases, protic and aprotic solvents, oils, and salt solutions. Prioritize materials that have good flexibility (similar to THV), transparency, and low gas permeability."
material_X = "elastomeric terpolymer of tetrafluoroethylene, hexafluoropropylene, and vinylidene fluoride"
application_Y = "broad chemical resistance"
keywords = [material_X, application_Y]

print("=" * 70)
print("Running System 1: Property Extraction")
print("=" * 70)
print(f"Query: {sentence}")
print(f"Material to replace (X): {material_X}")
print(f"Application (Y): {application_Y}")
print()

# Track System 1 timing
system1_start_time = datetime.utcnow()
pipeline_run["system1"]["start_time"] = system1_start_time.isoformat() + "Z"

# Run System 1 pipeline
system1_result = run_fixed_pipeline(
    sentence=sentence,
    keywords=keywords,
    analyst=analyst_s1,
    manager=manager_s1,
    research_assistant=research_assistant_s1,
    scientist=scientist_s1,  # Material Properties KG for Step 7
    pfas_scientist=pfas_scientist_s1,  # PFAS KG for Step 4 (question answering)
    include_rag_context=pipeline_cfg_s1["include_rag_context"],
    max_items=pipeline_cfg_s1["max_items"],
    temperature=config["llm"]["temperature"],
    n_results=pipeline_cfg_s1["n_results"],
    chat_logger=chat_logger_s1
)

# Track System 1 end time and duration
system1_end_time = datetime.utcnow()
system1_duration = (system1_end_time - system1_start_time).total_seconds()
pipeline_run["system1"]["end_time"] = system1_end_time.isoformat() + "Z"
pipeline_run["system1"]["duration_seconds"] = system1_duration

# Extract properties W and subgraph data
extracted_keywords = system1_result.get("extracted_keywords", []) or []
keywords_list = system1_result.get("keywords", []) or []
properties_W = {
    "required": extracted_keywords,
    "target_values": {}
}

# Extract constraints from System 1 (new: System 1 now extracts hard constraints)
extracted_constraints = system1_result.get("extracted_constraints", []) or []

# Extract subgraph data from keyword_connections for System 2
subgraph_data = None
keyword_connections = system1_result.get("keyword_connections")

if keyword_connections:
    # Serialize subgraph data for System 2
    subgraph_data = {
        "summary": keyword_connections.get("summary", {}) or {},
        "matched_node_ids": keyword_connections.get("matched_node_ids", []) or [],
        "found_paths": keyword_connections.get("found_paths", []) or [],
        "keyword_to_nodes": keyword_connections.get("keyword_to_nodes", {}) or {},
        "kg_results": keyword_connections.get("kg_results", {}) or {},
        "strategy": (keyword_connections.get("summary", {}) or {}).get("strategy", "single"),
    }

    # For connection_subgraph, serialize as node/edge lists
    connection_subgraph = keyword_connections.get("connection_subgraph")
    if connection_subgraph is not None:
        try:
            nodes = list(connection_subgraph.nodes())
            edges = list(connection_subgraph.edges())
            node_attributes = {n: dict(connection_subgraph.nodes[n]) for n in nodes}
            edge_attributes = {}
            is_multi = getattr(connection_subgraph, "is_multigraph", lambda: False)()
            if is_multi:
                for u, v, k, data in connection_subgraph.edges(keys=True, data=True):
                    edge_key = f"{u}|{v}|{k}"
                    edge_attributes[edge_key] = dict(data)
                edges = list(connection_subgraph.edges(keys=True))
            else:
                for u, v, data in connection_subgraph.edges(data=True):
                    edge_key = f"{u}|{v}"
                    edge_attributes[edge_key] = dict(data)

            subgraph_data["connection_subgraph"] = {
                "nodes": nodes,
                "edges": edges,
                "node_attributes": node_attributes,
                "edge_attributes": edge_attributes,
            }
        except Exception as e:
            print(f"Warning: Could not serialize connection_subgraph: {e}")
            subgraph_data["connection_subgraph"] = None

# Save System 1 output
timestamp = datetime.utcnow().isoformat() + "Z"
system1_run_id = chat_logger_s1.run_id
output_path = os.path.join(output_dir, f"system1_{system1_run_id}.json")

system1_payload = {
    "run_id": system1_run_id,
    "timestamp": timestamp,
    "sentence": system1_result.get("sentence"),
    "keywords": keywords_list,
    "material_X": material_X,
    "application_Y": application_Y,
    "properties_W": properties_W,
    "extracted_keywords": extracted_keywords,
    "extracted_constraints": extracted_constraints,
    "num_keywords": len(extracted_keywords),
    "num_constraints": len(extracted_constraints),
    "subgraph": subgraph_data,
    "chat_log_path": system1_result.get("chat_log_path"),
}

with open(output_path, "w", encoding="utf-8") as f:
    json.dump(system1_payload, f, indent=2, ensure_ascii=False, default=str)

# Update pipeline_run with System 1 results
pipeline_run["system1"]["run_id"] = system1_run_id
pipeline_run["system1"]["result_path"] = output_path
pipeline_run["system1"]["chat_log_path"] = system1_result.get("chat_log_path")
pipeline_run["system1"]["properties_extracted"] = len(extracted_keywords)

print(f"\n✓ System 1 complete!")
print(f"✓ Saved properties W and subgraph to {output_path}")
print(f"✓ Extracted {len(extracted_keywords)} properties")
print(f"✓ System 1 duration: {system1_duration:.2f} seconds")

Running System 1: Property Extraction
Query: Find a substitute material for PFAS material THV (elastomeric terpolymer of tetrafluoroethylene, hexafluoropropylene, and vinylidene fluoride) that has broad chemical resistance, including acids, bases, protic and aprotic solvents, oils, and salt solutions. Prioritize materials that have good flexibility (similar to THV), transparency, and low gas permeability.
Material to replace (X): elastomeric terpolymer of tetrafluoroethylene, hexafluoropropylene, and vinylidene fluoride
Application (Y): broad chemical resistance

Material Requirements Analysis: Starting Process

----------------------------------------------------------------------
Step 1: ResearchAnalyst - Performing RAG Retrieval
----------------------------------------------------------------------
→ Querying ChromaDB with sentence and keywords...
✓ RAG retrieval complete: 0 documents retrieved
✗ No RAG results found, using analyze_question instead (i.e., no keywords used in retriev

In [10]:
# Initialize loop variables
max_iterations = 3
iteration = 0
all_iterations = []  # Track all iterations for summary
# Constraints from System 1 (extracted hard constraints + any user-specified ones)
constraints_U = list(extracted_constraints)  # populated by System 1's extract_constraints step
print(f"✓ Constraints from System 1: {len(constraints_U)}")
if constraints_U:
    for i, c in enumerate(constraints_U[:5], 1):
        print(f"  {i}. {c}")
    if len(constraints_U) > 5:
        print(f"  ... and {len(constraints_U) - 5} more")

# Cache for Step 1 result from first iteration (to avoid re-running expensive operations)
cached_substitution_result = None

# Extract base run_id from System 1
system1_run_id_base = system1_run_id.split("_")[0] if "_" in system1_run_id else system1_run_id

# Initialize pipeline_run loop metadata
pipeline_run["system2_system3_loop"]["max_iterations"] = max_iterations

print("=" * 70)
print("Starting System 2 → System 3 Loop")
print("=" * 70)
print(f"Max iterations: {max_iterations}")
print(f"Base run_id: {system1_run_id_base}")
print()

while iteration < max_iterations:
    iteration += 1
    print("\n" + "=" * 70)
    print(f"Iteration {iteration}/{max_iterations}")
    print("=" * 70)
    
    # Generate run_id for this iteration
    pattern_s2 = os.path.join(output_dir, f"system2_{system1_run_id_base}_*.json")
    existing_s2 = glob.glob(pattern_s2)
    max_counter_s2 = -1
    for f in existing_s2:
        try:
            n = int(os.path.basename(f).replace(".json", "").split("_")[-1])
            max_counter_s2 = max(max_counter_s2, n)
        except (ValueError, IndexError):
            pass
    system2_counter = max_counter_s2 + 1
    system2_run_id = f"{system1_run_id_base}_{system2_counter}"
    
    # Create chat logger for System 2
    chat_logger_s2 = ChatLogger(
        run_id=system2_run_id,
        pipeline="material_discovery"
    )
    
    # Update System 2 agents with chat logger
    analyst_patents_s2.chat_logger = chat_logger_s2
    analyst_materialdb_s2.chat_logger = chat_logger_s2
    scientist_s2.chat_logger = chat_logger_s2
    
    # Create MultiAnalyst wrapper for System 2
    analyst_s2 = MultiAnalyst({
        "patents": analyst_patents_s2,
        "materialdb": analyst_materialdb_s2
    })
    
    # Create manager for System 2
    manager_s2 = ResearchManager(
        name="research_manager",
        system_message=None,
        generate_fn=generate,
        chat_logger=chat_logger_s2
    )
    
    # Create MaterialScientist for System 2
    material_scientist_s2 = MaterialScientist(
        material_db=material_db,
        knowledge_graph=G_materialproperties,
        analyst=analyst_s2,
        manager=manager_s2,
        embedding_model=embedding_model,
        embedding_tokenizer=embedding_tokenizer
    )
    
    print(f"\n--- Running System 2 (run_id: {system2_run_id}) ---")
    
    # Track System 2 timing
    system2_start_time = datetime.utcnow()
    
    # Run System 2
    system2_result = run_material_discovery_pipeline(
        material_X=material_X,
        application_Y=application_Y,
        properties_W=properties_W,
        constraints_U=constraints_U,
        analyst=analyst_s2,
        manager=manager_s2,
        scientist=scientist_s2,
        tracker=tracker,  # Shared tracker - will avoid previously rejected candidates
        max_iterations=pipeline_cfg_s2["max_iterations"],
        temperature=config["llm"]["temperature"],
        chat_logger=chat_logger_s2,
        subgraph_data=subgraph_data,
        material_db=material_db,
        subgraph_processor=subgraph_processor,
        material_grounding=material_grounding,
        material_scientist=material_scientist_s2,
        knowledge_graph=G_materialproperties,
        substitution_result=cached_substitution_result  # Use cached Step 1 result if available
    )
    
    # Track System 2 end time and duration
    system2_end_time = datetime.utcnow()
    system2_duration = (system2_end_time - system2_start_time).total_seconds()
    
    # Extract internal iteration timing from System 2
    internal_iterations = []
    iteration_history = system2_result.get("iteration_history", [])
    if iteration_history and system2_result.get("chat_log_path"):
        # Try to get timing from chat log
        try:
            with open(system2_result.get("chat_log_path"), "r") as f:
                chat_log = json.load(f)
            chat_start = datetime.fromisoformat(chat_log.get("timestamp", "").replace("Z", "+00:00"))
            interactions = chat_log.get("interactions", [])
            
            # Group interactions by iteration (approximate)
            # This is a simplified approach - in practice, you might need more sophisticated parsing
            for i, hist_item in enumerate(iteration_history):
                iter_num = hist_item.get("iteration", i + 1)
                candidate = hist_item.get("candidate", {}).get("material_name", "Unknown")
                status = "accepted" if hist_item.get("feasible") else "rejected"
                
                # Approximate timing - use chat log start + proportional time
                # This is a rough estimate; for precise timing, you'd need timestamps in iteration_history
                if len(iteration_history) > 0:
                    iter_start_offset = (system2_duration / len(iteration_history)) * (iter_num - 1)
                    iter_end_offset = (system2_duration / len(iteration_history)) * iter_num
                    iter_start = system2_start_time + timedelta(seconds=iter_start_offset)
                    iter_end = system2_start_time + timedelta(seconds=iter_end_offset)
                else:
                    iter_start = system2_start_time
                    iter_end = system2_end_time
                
                internal_iterations.append({
                    "iteration": iter_num,
                    "start_time": iter_start.isoformat() + "Z",
                    "end_time": iter_end.isoformat() + "Z",
                    "duration_seconds": (iter_end - iter_start).total_seconds(),
                    "candidate": candidate,
                    "status": status
                })
        except Exception as e:
            print(f"Warning: Could not extract internal iteration timing: {e}")
    
    # Save System 2 result
    timestamp_s2 = datetime.utcnow().isoformat() + "Z"
    system2_output_path = os.path.join(output_dir, f"system2_{system2_run_id}.json")
    
    discovery_payload = {
        "run_id": system2_run_id,
        "timestamp": timestamp_s2,
        "application": application_Y,
        "properties": properties_W,
        "constraints": constraints_U,
        "success": system2_result.get("success", False),
        "candidate": system2_result.get("candidate"),
        "iterations": system2_result.get("iterations", 0),
        "rejected_candidates": system2_result.get("rejected_candidates", []),
        "final_constraints": system2_result.get("final_constraints", []),
        "evidence_summary": system2_result.get("evidence_summary", {}),
        "iteration_history": system2_result.get("iteration_history", []),
        "property_mapping": system2_result.get("property_mapping", {}),
        "chat_log_path": system2_result.get("chat_log_path")
    }
    
    with open(system2_output_path, "w") as f:
        json.dump(discovery_payload, f, indent=2)
    
    print(f"✓ System 2 complete - saved to {system2_output_path}")
    print(f"✓ System 2 duration: {system2_duration:.2f} seconds")
    
    # Cache Step 1 result from first iteration for reuse in subsequent iterations
    if iteration == 1 and cached_substitution_result is None:
        cached_substitution_result = system2_result.get("substitution_result")
        if cached_substitution_result:
            print(f"✓ Cached Step 1 result for reuse in subsequent iterations")
    
    if not system2_result.get("success"):
        print(f"✗ System 2 failed to find a candidate material")
        all_iterations.append({
            "iteration": iteration,
            "system2_run_id": system2_run_id,
            "system2_success": False,
            "system3_run_id": None,
            "system3_status": None
        })
        break
    
    candidate_name = system2_result.get("candidate", {}).get("material_name", "Unknown")
    print(f"✓ System 2 found candidate: {candidate_name}")
    
    # Track System 3 timing
    system3_start_time = datetime.utcnow()
    
    # Generate run_id for System 3
    pattern_s3 = os.path.join(output_dir, f"system3_{system1_run_id_base}_*.json")
    existing_s3 = glob.glob(pattern_s3)
    max_counter_s3 = -1
    for f in existing_s3:
        try:
            n = int(os.path.basename(f).replace(".json", "").split("_")[-1])
            max_counter_s3 = max(max_counter_s3, n)
        except (ValueError, IndexError):
            pass
    system3_counter = max_counter_s3 + 1
    system3_run_id = f"{system1_run_id_base}_{system3_counter}"
    
    # Create chat logger for System 3
    chat_logger_s3 = ChatLogger(
        run_id=system3_run_id,
        pipeline="manufacturability_assessment"
    )
    
    # Create manager for System 3
    manager_s3 = ResearchManager(
        name="research_manager",
        system_message=None,
        generate_fn=generate,
        chat_logger=chat_logger_s3
    )
    
    print(f"\n--- Running System 3 (run_id: {system3_run_id}) ---")
    
    # Run System 3
    system3_result = run_manufacturability_assessment_pipeline(
        system2_result=system2_result,
        initial_query=sentence,
        material_X=material_X,
        application_Y=application_Y,
        constraints_U=constraints_U,
        tracker=tracker,  # Shared tracker - System 3 will add rejections here
        process_analyst=process_analyst,
        manager=manager_s3,
        properties_W=properties_W,
        config=config,
        temperature=0,
        chat_logger=chat_logger_s3,
    )
    
    # Track System 3 end time and duration
    system3_end_time = datetime.utcnow()
    system3_duration = (system3_end_time - system3_start_time).total_seconds()
    
    # Save System 3 result
    timestamp_s3 = datetime.utcnow().isoformat() + "Z"
    system3_output_path = os.path.join(output_dir, f"system3_{system3_run_id}.json")
    
    with open(system3_output_path, "w", encoding="utf-8") as f:
        system3_result["run_id"] = system3_run_id
        json.dump(system3_result, f, indent=2, default=str)
    
    print(f"✓ System 3 complete - saved to {system3_output_path}")
    print(f"✓ System 3 duration: {system3_duration:.2f} seconds")
    
    # Check status
    status = system3_result.get("status", "")
    manufacturable = system3_result.get("manufacturable", False)
    
    print(f"\nSystem 3 Status: {status.upper()}")
    
    # Record iteration in all_iterations
    all_iterations.append({
        "iteration": iteration,
        "system2_run_id": system2_run_id,
        "system2_success": True,
        "candidate": candidate_name,
        "system3_run_id": system3_run_id,
        "system3_status": status,
        "manufacturable": manufacturable
    })
    
    # Record iteration in pipeline_run
    iteration_data = {
        "iteration": iteration,
        "system2": {
            "run_id": system2_run_id,
            "start_time": system2_start_time.isoformat() + "Z",
            "end_time": system2_end_time.isoformat() + "Z",
            "duration_seconds": system2_duration,
            "result_path": system2_output_path,
            "chat_log_path": system2_result.get("chat_log_path"),
            "candidate": candidate_name,
            "internal_iterations": internal_iterations
        },
        "system3": {
            "run_id": system3_run_id,
            "start_time": system3_start_time.isoformat() + "Z",
            "end_time": system3_end_time.isoformat() + "Z",
            "duration_seconds": system3_duration,
            "result_path": system3_output_path,
            "chat_log_path": system3_result.get("chat_log_path"),
            "status": status,
            "blocking_constraints": system3_result.get("blocking_constraints", []),
            "feedback_to_system2": system3_result.get("feedback_to_system2", "")
        }
    }
    pipeline_run["system2_system3_loop"]["iterations"].append(iteration_data)
    pipeline_run["system2_system3_loop"]["total_iterations"] = len(pipeline_run["system2_system3_loop"]["iterations"])
    
    # Check if manufacturable - if so, break loop
    if status == "manufacturable":
        print("\n" + "=" * 70)
        print("✓ SUCCESS: Found manufacturable candidate!")
        print("=" * 70)
        break
    
    # If blocked, continue to next iteration
    if status == "blocked":
        feedback = system3_result.get("feedback_to_system2", "")
        print(f"\n✗ Candidate rejected (blocked)")
        print(f"Feedback to System 2: {feedback[:200]}...")
        print(f"Tracker updated - rejected candidates: {len(tracker.get_all_rejected())}")

        # Persist structured System 3 feedback history for auditability.
        feedback_history = pipeline_run["system2_system3_loop"].setdefault("system3_feedback_history", [])
        blocking = system3_result.get("blocking_constraints", [])
        feedback_history.append({
            "iteration": iteration,
            "system3_run_id": system3_run_id,
            "blocking_constraints": blocking,
            "feedback_to_system2": feedback,
        })

        # Compact + deduplicate feedback before feeding it to the next System 2 iteration.
        existing_constraints_norm = {str(c).strip().lower() for c in constraints_U if str(c).strip()}
        compact_constraints = []
        for bc in blocking:
            if isinstance(bc, dict):
                ctype = str(bc.get("type", "missing_critical_info")).strip()
                desc = str(bc.get("description", "")).strip()
            else:
                ctype = "missing_critical_info"
                desc = str(bc).strip()
            if not desc:
                continue
            compact = f"S3[{ctype}]: {desc[:220]}"
            if compact.lower() not in existing_constraints_norm:
                compact_constraints.append(compact)
                existing_constraints_norm.add(compact.lower())

        if feedback:
            feedback_compact = f"S3 feedback: {feedback[:220]}"
            if feedback_compact.lower() not in existing_constraints_norm:
                compact_constraints.append(feedback_compact)
                existing_constraints_norm.add(feedback_compact.lower())

        constraints_U.extend(compact_constraints)
        print(f"  → Added {len(compact_constraints)} compact System 3 constraints")
        print(f"  → Updated constraints_U: {len(constraints_U)} total constraints")

        if iteration < max_iterations:
            print(f"\n→ Looping back to System 2 for iteration {iteration + 1}...")
        else:
            print(f"\n→ Max iterations ({max_iterations}) reached")

print("\n" + "=" * 70)
print("System 2 → System 3 Loop Complete")
print("=" * 70)
print(f"Total iterations: {len(all_iterations)}")
print(f"Final status: {all_iterations[-1]['system3_status'] if all_iterations else 'N/A'}")

✓ Constraints from System 1: 11
  1. Hard Constraints for a THV Substitute
  2. Acid resistance – Must lose ≤ 0.5 % of its mass after 100 h exposure to 1 M H₂SO₄ at 80 °C.
  3. Base resistance – Must swell (dimensional change) by ≤ 2 % after 24 h exposure to 1 M NaOH at 25 °C.
  4. Protic‑solvent resistance – Must change dimensions by ≤ 3 % after 48 h exposure to methanol or ethanol at 40 °C.
  5. Aprotic‑solvent resistance – Must lose ≤ 1 % of its mass after 72 h exposure to acetone or DMSO at 60 °C.
  ... and 6 more
Starting System 2 → System 3 Loop
Max iterations: 3
Base run_id: 2026031908


Iteration 1/3

--- Running System 2 (run_id: 2026031908_0) ---
Material Discovery Pipeline: Starting Process

Pipeline Configuration:
  [OK] Agents verified
  [OK] Material to replace (X): elastomeric terpolymer of tetrafluoroethylene, hexafluoropropylene, and vinylidene fluoride
  [OK] Application (Y): broad chemical resistance
  [OK] Required Properties: ['Key material properties & requirement

In [11]:
print("=" * 70)
print("HIERARCHICAL SYSTEM EXECUTION SUMMARY")
print("=" * 70)

print(f"\nSystem 1 (Property Extraction):")
print(f"  Run ID: {system1_run_id}")
print(f"  Properties extracted: {len(properties_W.get('required', []))}")
print(f"  Output: {os.path.join(output_dir, f'system1_{system1_run_id}.json')}")

print(f"\nSystem 2 → System 3 Loop:")
print(f"  Total iterations: {len(all_iterations)}")
print(f"  Max iterations allowed: {max_iterations}")

for iter_data in all_iterations:
    iter_num = iter_data["iteration"]
    s2_run_id = iter_data["system2_run_id"]
    s2_success = iter_data["system2_success"]
    candidate = iter_data.get("candidate", "N/A")
    s3_run_id = iter_data.get("system3_run_id", "N/A")
    s3_status = iter_data.get("system3_status", "N/A")
    
    print(f"\n  Iteration {iter_num}:")
    print(f"    System 2: {'✓ Success' if s2_success else '✗ Failed'} (run_id: {s2_run_id})")
    if s2_success:
        print(f"      Candidate: {candidate}")
        print(f"    System 3: {s3_status.upper()} (run_id: {s3_run_id})")
        if s3_status == "blocked":
            # Load System 3 result to show feedback
            s3_path = os.path.join(output_dir, f"system3_{s3_run_id}.json")
            if os.path.exists(s3_path):
                with open(s3_path, "r") as f:
                    s3_data = json.load(f)
                feedback = s3_data.get("feedback_to_system2", "")
                if feedback:
                    print(f"      Feedback: {feedback[:150]}...")

print(f"\nFinal Result:")
if all_iterations:
    final_iter = all_iterations[-1]
    final_status = final_iter.get("system3_status", "N/A")
    final_candidate = final_iter.get("candidate", "N/A")
    
    if final_status == "manufacturable":
        print(f"  ✓ SUCCESS: Found manufacturable candidate!")
        print(f"  Candidate: {final_candidate}")
        print(f"  System 3 Run ID: {final_iter.get('system3_run_id', 'N/A')}")
        
        # Load and display process recipe if available
        s3_path = os.path.join(output_dir, f"system3_{final_iter.get('system3_run_id', '')}.json")
        if os.path.exists(s3_path):
            with open(s3_path, "r") as f:
                s3_data = json.load(f)
            recipe = s3_data.get("process_recipe", [])
            if recipe:
                print(f"\n  Process Recipe ({len(recipe)} steps):")
                for step in recipe[:3]:  # Show first 3 steps
                    step_idx = step.get("step_index", "?")
                    desc = step.get("description", "")[:100]
                    print(f"    Step {step_idx}: {desc}...")
                if len(recipe) > 3:
                    print(f"    ... and {len(recipe) - 3} more steps")
    elif final_status == "blocked":
        print(f"  ✗ BLOCKED: Candidate rejected after {len(all_iterations)} iteration(s)")
        print(f"  Final candidate: {final_candidate}")
        if len(all_iterations) >= max_iterations:
            print(f"  Max iterations ({max_iterations}) reached")
    else:
        print(f"  Status: {final_status}")
else:
    print("  No iterations completed")

print(f"\nTracker Status:")
rejected = tracker.get_all_rejected()
print(f"  Total rejected candidates: {len(rejected)}")
if rejected:
    print(f"  Rejected materials:")
    for i, mat in enumerate(rejected[:5], 1):
        print(f"    {i}. {mat}")
    if len(rejected) > 5:
        print(f"    ... and {len(rejected) - 5} more")

print("\n" + "=" * 70)

# Update final outcome in pipeline_run
if all_iterations:
    final_iter = all_iterations[-1]
    final_status = final_iter.get("system3_status", "N/A")
    final_candidate = final_iter.get("candidate", "N/A")
    
    if final_status == "manufacturable":
        pipeline_run["final_outcome"]["status"] = "manufacturable"
    elif final_status == "blocked":
        if len(all_iterations) >= max_iterations:
            pipeline_run["final_outcome"]["status"] = "max_iterations_reached"
        else:
            pipeline_run["final_outcome"]["status"] = "blocked"
    else:
        pipeline_run["final_outcome"]["status"] = final_status
    
    pipeline_run["final_outcome"]["final_candidate"] = final_candidate
else:
    pipeline_run["final_outcome"]["status"] = "no_iterations_completed"

pipeline_run["final_outcome"]["total_rejected_candidates"] = len(tracker.get_all_rejected())

# Calculate total duration and update pipeline_run
pipeline_end_time = datetime.utcnow()
pipeline_total_duration = (pipeline_end_time - pipeline_start_time).total_seconds()
pipeline_run["end_time"] = pipeline_end_time.isoformat() + "Z"
pipeline_run["total_duration_seconds"] = pipeline_total_duration

# Save PipelineRun object
pipeline_run_output_path = os.path.join(output_dir, f"pipeline_run_{pipeline_run_id}.json")
with open(pipeline_run_output_path, "w", encoding="utf-8") as f:
    json.dump(pipeline_run, f, indent=2, ensure_ascii=False, default=str)

print(f"\n✓ PipelineRun object saved to {pipeline_run_output_path}")
print(f"✓ Total pipeline duration: {pipeline_total_duration:.2f} seconds ({pipeline_total_duration/60:.2f} minutes)")

# Save evaluation-ready JSON for material scientist evaluation
eval_path = save_evaluation_export(pipeline_run, output_dir)
if eval_path:
    print(f"✓ Evaluation export saved to {eval_path}")

HIERARCHICAL SYSTEM EXECUTION SUMMARY

System 1 (Property Extraction):
  Run ID: 2026031908_0
  Properties extracted: 38
  Output: /orcd/pool/004/tphage/SG_march/MARS/pipeline_logs/system1_2026031908_0.json

System 2 → System 3 Loop:
  Total iterations: 2
  Max iterations allowed: 3

  Iteration 1:
    System 2: ✓ Success (run_id: 2026031908_0)
      Candidate: PEEK/Transparent Polyurethane Elastomer Composite
    System 3: BLOCKED (run_id: 2026031908_0)
      Feedback: Exclude candidates that require simultaneous high‑temperature processing of incompatible polymers without proven lab‑scale equipment or clear mitigati...

  Iteration 2:
    System 2: ✓ Success (run_id: 2026031908_1)
      Candidate: Engage™ 030902 HEALTH+ Polyolefin Elastomer / High‑Modulus Transparent TPU Composite
    System 3: MANUFACTURABLE (run_id: 2026031908_1)

Final Result:
  ✓ SUCCESS: Found manufacturable candidate!
  Candidate: Engage™ 030902 HEALTH+ Polyolefin Elastomer / High‑Modulus Transparent TPU Compos